In [2]:
from loguru import logger
from typing import TypedDict,Literal
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
from langgraph.constants import START, END
from langgraph.graph import StateGraph

load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)

# 1. 定义状态
class OverAllState(TypedDict):
    topic: str
    poem: str
    joke: str
    content_type:str

# 2. 定义节点
def node_a(state: OverAllState) -> OverAllState:
    response = model.invoke([HumanMessage(content=f"写一首关于{state['topic']}的唐诗")])
    return {"poem": response.content}

def node_b(state: OverAllState) -> OverAllState:
    response = model.invoke([HumanMessage(content=f"写一个关于{state['topic']}的笑话")])
    return {"joke": response.content}
def audit_node(state: OverAllState) -> OverAllState:
    logger.info(f"任务阶段已经全部执行完毕,诗{'已生成' if state["poem"] else '未生成'},笑话{'已生成' if state["joke"] else '未生成'}")


# 3. 构建图
builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_a", node_a)
builder.add_node("node_b", node_b)
builder.add_node("audit_node", audit_node,defer=True)

builder.add_edge(START, "node_a")
builder.add_edge(START, "node_b")
builder.add_edge(START, "audit_node")
builder.add_edge("node_a", END)
builder.add_edge("node_b", END)
builder.add_edge("audit_node", END)

graph = builder.compile()
graph.invoke({"topic": "猫猫", "content_type": "诗"})

2026-08-10 14:11:41.346 | INFO     | __main__:audit_node:36 - 任务阶段已经全部执行完毕,诗已生成,笑话已生成


{'topic': '猫猫',
 'poem': '《咏猫》\n乌圆雪爪步轻尘，碧眼窥人欲近身。\n午枕花间眠未稳，忽惊蝶影过东邻。\n\n注：本诗以唐代咏物诗风格为基，摹写狸奴灵动态。首联“乌圆”化用古人呼猫雅称，“雪爪”点染其形，碧眼窥人写尽猫儿狡黠。尾联以蝶影惊眠收束，暗合“猫扑流萤”之趣，深得晚唐小景诗神韵。',
 'joke': '一只猫走进一家便利店，问店员：“你们这里有卖老鼠吗？”  \n店员说：“有啊，在后面冰柜里。”  \n猫走出去，第二天又来了：“你们还有老鼠吗？”  \n店员：“有，今天刚进的。”  \n猫又走了。  \n第三天，猫再来，店员忍不住问：“你光问不买，到底想干嘛？”  \n猫舔舔爪子：“我就是想确认一下，那老鼠是冷冻食品，还是活物。”  \n店员一愣：“……活物怎么了？”  \n猫咧嘴一笑：“活物的话，我昨晚上家门口已经收到三只了——我妈说，这叫网购退货。”',
 'content_type': '诗'}